In [13]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from pycocotools.coco import COCO
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
import time
from tqdm import tqdm
from baseline_model import UNet
from dataset import COCOSegmentationDataset
from torchmetrics.segmentation import MeanIoU

In [14]:
# Dice Loss implementation
class DiceLoss(torch.nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
        
    def forward(self, logits, targets):
        # Apply sigmoid to get probabilities
        probs = torch.sigmoid(logits)
        
        # Flatten predictions and targets
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        
        # Calculate intersection and union
        intersection = (probs_flat * targets_flat).sum()
        union = probs_flat.sum() + targets_flat.sum()
        
        # Calculate Dice coefficient
        dice_coeff = (2. * intersection + self.smooth) / (union + self.smooth)
        
        # Return Dice loss (1 - Dice coefficient)
        return 1.0 - dice_coeff


In [15]:
def visualize_predictions(model, data_loader, output_dir="val_plots", device="cuda", num_samples=5):
    """
    Visualizes model predictions on validation samples, showing only the segmented region.
    Areas outside the segmented region will appear black.
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

    # Get sample batches
    samples = []
    for i, (data, mask) in enumerate(data_loader):
        if i >= num_samples:
            break
        samples.append((data, mask))

    # Generate and save plots
    model.eval()
    with torch.no_grad():
        for i, (data, mask) in enumerate(samples):
            # Move data to device and predict
            data = data.to(device)
            mask = mask.to(device)
            output = model(data)
            pred = (torch.sigmoid(output) > 0.5).float()

            # Convert to CPU numpy
            input_np = data[0].permute(1, 2, 0).cpu().numpy()
            true_np = mask[0, 0].cpu().numpy()
            pred_np = pred[0, 0].cpu().numpy()

            # Create figure
            plt.figure(figsize=(15, 5))

            # Input image
            plt.subplot(1, 3, 1)
            plt.title("Input")
            plt.imshow(input_np)
            plt.axis('off')

            # Ground truth
            plt.subplot(1, 3, 2)
            plt.title("Ground Truth")
            plt.imshow(true_np, cmap='gray')
            plt.axis('off')

            # Prediction
            plt.subplot(1, 3, 3)
            plt.title("Prediction")
            plt.imshow(pred_np, cmap='gray')
            plt.axis('off')

            plt.tight_layout()
            plt.savefig(f"{output_dir}/sample_{i}.png", bbox_inches='tight', pad_inches=1)
            plt.close()
            
            
    print(f"Saved {num_samples} prediction visualizations to {output_dir}")

In [16]:
def plot_training_history(history):
    plt.figure(figsize=(15, 5))
    
    # Loss subplot
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train Loss', marker="o")
    plt.plot(history['val_loss'], label='Validation Loss', marker="o")
    plt.title('Loss History')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Dice subplot
    plt.subplot(1, 3, 2)
    plt.plot(history['train_dice'], label='Train Dice', marker=".")
    plt.plot(history['val_dice'], label='Validation Dice', marker=".")
    plt.title('Dice Coefficient History')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Coefficient')
    plt.legend()
    
    # IoU subplot
    plt.subplot(1, 3, 3)
    plt.plot(history['train_iou'], label='Train IoU', marker=".")
    plt.plot(history['val_iou'], label='Validation IoU', marker=".")
    plt.title('Mean IoU History')
    plt.xlabel('Epoch')
    plt.ylabel('Mean IoU')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('./visualizations/train_plots/diceloss/training_history.png')
    plt.close()


In [17]:
# Dice coefficient calculation
def dice_coeff(pred, target):
    smooth = 1.0
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return (2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)

In [23]:
def load_model(model_path, device="cuda"):
    """Load a saved model from checkpoint"""
    # Initialize model architecture
    model = UNet(in_channels=3, out_classes=1)
    model = model.to(device)
    
    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    
    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load other training information if available
    history = checkpoint.get('history', None)
    test_metrics = checkpoint.get('test_metrics', None)
    
    print(f"Loaded model from {model_path}")
    if history:
        print(f"Model was trained for {len(history['train_loss'])} epochs")
        print(f"Best validation loss: {min(history['val_loss']):.4f}")
        if 'train_dice' in history and 'val_dice' in history:
            print(f"Best training Dice: {max(history['train_dice']):.4f}")
            print(f"Best validation Dice: {max(history['val_dice']):.4f}")
        if 'train_iou' in history and 'val_iou' in history:
            print(f"Best training IoU: {max(history['train_iou']):.4f}")
            print(f"Best validation IoU: {max(history['val_iou']):.4f}")
    if test_metrics:
        print(f"Test metrics - Loss: {test_metrics['test_loss']:.4f}, Dice: {test_metrics['test_dice']:.4f}, IoU: {test_metrics.get('test_iou', 0):.4f}")
    
    return model, history

In [19]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, model_path, visualization_path, model_pred_path, training_run, num_epochs=25, device="cuda"):
    model = model.to(device)
    best_loss = float('inf')

    
    # Create directories for checkpoints and visualizations
    os.makedirs(model_path, exist_ok=True)
    os.makedirs(visualization_path, exist_ok=True)
    
    # Training metrics history
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_dice': [],
        'val_dice': [],
        'train_iou': [],
        'val_iou': []
    }
    
    # Initialize Mean IoU metrics for both training and validation
    # For binary segmentation, num_classes=2 (background and foreground)
    train_iou_metric = MeanIoU(num_classes=2)
    train_iou_metric.to(device)
    val_iou_metric = MeanIoU(num_classes=2)
    val_iou_metric.to(device)
    
    # Training loop
    for epoch in range(num_epochs):
        start_time = time.time()
        model.train()
        train_loss = 0
        train_dice = 0
        train_iou_metric.reset()  # Reset metrics at the start of each epoch
        
        # Training phase
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for batch_idx, (data, target) in enumerate(train_loop):
            data, target = data.to(device), target.to(device).float()  # Convert to float for dice loss
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            # Calculate metrics
            train_loss += loss.item()
            train_dice += dice_coeff(output, target).item()
            
            # For IoU metric, convert sigmoid outputs to binary predictions
            preds = (torch.sigmoid(output) > 0.5).long()
            targets = target.long()  # Convert target to long for IoU metric
            
            # Update IoU metric (expects class indices, not probabilities)
            train_iou_metric.update(preds, targets)
            
            # Update progress bar
            current_dice = dice_coeff(output, target).item()
            train_loop.set_postfix(loss=loss.item(), dice=current_dice)
        
        # Calculate average metrics
        train_loss /= len(train_loader)
        train_dice /= len(train_loader)
        train_iou = train_iou_metric.compute().item()  # Compute IoU for the epoch
        
        history['train_loss'].append(train_loss)
        history['train_dice'].append(train_dice)
        history['train_iou'].append(train_iou)
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_dice = 0
        val_iou_metric.reset()  # Reset validation IoU metric
        
        with torch.no_grad():
            val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
            for batch_idx, (data, target) in enumerate(val_loop):
                data, target = data.to(device), target.to(device).float()
                output = model(data)
                loss = criterion(output, target)
                
                # Calculate metrics
                val_loss += loss.item()
                val_dice += dice_coeff(output, target).item()
                
                # For IoU metric
                preds = (torch.sigmoid(output) > 0.5).long()
                targets = target.long()
                
                # Update IoU metric
                val_iou_metric.update(preds, targets)
                
                # Update progress bar
                current_dice = dice_coeff(output, target).item()
                val_loop.set_postfix(loss=loss.item(), dice=current_dice)
        
        # Calculate average metrics
        val_loss /= len(val_loader)
        val_dice /= len(val_loader)
        val_iou = val_iou_metric.compute().item()  # Compute IoU for validation
        
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)
        history['val_iou'].append(val_iou)
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Calculate epoch time
        epoch_time = time.time() - start_time
        
        # Print epoch summary
        print(f"Epoch {epoch+1}/{num_epochs} completed in {epoch_time:.2f}s")
        print(f"Train Loss: {train_loss:.4f}, Train Dice coeff: {train_dice:.4f}, Train IoU: {train_iou:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val Dice coeff: {val_dice:.4f}, Val IoU: {val_iou:.4f}")
        
        # Visualize predictions on validation set
        # if (epoch + 1) % 5 == 0 or epoch == num_epochs - 1:
        #     visualize_predictions(model, val_loader, 
        #                          output_dir=f"visualizations/predictions/dice_loss", 
        #                          device=device,
        #                          num_samples=5)
        
        # Save checkpoint if this is the best model so far
        if val_loss < best_loss:
            
            visualize_predictions(model, val_loader, 
                                 output_dir=f'{model_pred_path}', 
                                 device=device,
                                 num_samples=5)
            
            best_loss = val_loss
            torch.save({
            'model_state_dict': model.state_dict(),
            'history': history
            }, f"{model_path}{training_run}.pth")
            print(f"Saved new best model with val_loss: {val_loss:.4f}")
        
    # Plot and save training history
    plot_training_history(history)
    
    return model, history

In [20]:
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Data paths
train_img_dir = "./dataset_phase_1/segmentation_dataset/seg_train/images/"
train_ann_file = "./dataset_phase_1/segmentation_dataset/seg_train/annotations/seg_train.json"
val_img_dir = "./dataset_phase_1/segmentation_dataset/seg_val/images/"
val_ann_file = "./dataset_phase_1/segmentation_dataset/seg_val/annotations/seg_val.json"

model_path = "models/diceloss/"
visualization_path = "visualizations/train_plots/diceloss/"
model_pred_path = "visualizations/predictions/diceloss"



# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Initialize COCO API
train_coco = COCO(train_ann_file)
val_coco = COCO(val_ann_file)


# Create datasets
image_size = (512, 512) # as base size of image is same, we can do resize as well
batch_size = 4

train_dataset = COCOSegmentationDataset(
    coco=train_coco,
    img_dir=train_img_dir,
    image_size=image_size
)

val_dataset = COCOSegmentationDataset(
    coco=val_coco,
    img_dir=val_img_dir,
    image_size=image_size
)

print(f"Train dataset contains: {len(train_dataset)} samples")
print(f"Validation dataset contains: {len(val_dataset)} samples")


Using device: cuda
loading annotations into memory...
Done (t=0.17s)
creating index...
index created!
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
Train dataset contains: 1000 samples
Validation dataset contains: 200 samples


In [12]:
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=16
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=16
)

# Initialize model
model = UNet(in_channels=3, out_classes=1) # as image has 3 channels

# Define Dice loss function instead of BCE
criterion = DiceLoss(smooth=1.0)

# Define optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5) 
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-5)


# Define scheduler for learning rate adjustment
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
num_epochs = 20
training_run = 12

# Train model
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    model_path = model_path,
    visualization_path = visualization_path,
    model_pred_path = model_pred_path,
    training_run=training_run,
    num_epochs=num_epochs,
    device=device)
print("Training complete!")

Epoch 1/20 [Val]: 100%|██████████| 50/50 [00:04<00:00, 10.05it/s, dice=0.153, loss=0.847]

Epoch 1/20 completed in 19.58s
Train Loss: 0.8749, Train Dice coeff: 0.1251, Train IoU: 0.2126
Val Loss: 0.8492, Val Dice coeff: 0.1508, Val IoU: 0.2498


Saved 5 prediction visualizations to visualizations/predictions/diceloss
Saved new best model with val_loss: 0.8492


Epoch 2/20 [Val]: 100%|██████████| 50/50 [00:03<00:00, 12.90it/s, dice=0.178, loss=0.822]

Epoch 2/20 completed in 26.69s
Train Loss: 0.8401, Train Dice coeff: 0.1599, Train IoU: 0.3264
Val Loss: 0.8209, Val Dice coeff: 0.1791, Val IoU: 0.3412


...

Epoch 19/20 [Val]: 100%|██████████| 50/50 [00:04<00:00, 11.67it/s, dice=0.574, loss=0.426]


Epoch 19/20 completed in 29.65s
Train Loss: 0.1818, Train Dice coeff: 0.8182, Train IoU: 0.7508
Val Loss: 0.2756, Val Dice coeff: 0.7244, Val IoU: 0.6126


Epoch 20/20 [Val]: 100%|██████████| 50/50 [00:03<00:00, 12.91it/s, dice=0.71, loss=0.29]  

Epoch 20/20 completed in 29.19s
Train Loss: 0.1693, Train Dice coeff: 0.8307, Train IoU: 0.7588
Val Loss: 0.2535, Val Dice coeff: 0.7465, Val IoU: 0.6342


Saved 5 prediction visualizations to visualizations/predictions/diceloss
Saved new best model with val_loss: 0.2535
Training complete!


In [25]:
training_run = 12
load_model_path = f"models/diceloss/{training_run}.pth"  # Path to your saved model
# Load existing model
model, history = load_model(load_model_path, device)

# Use DiceLoss instead of BCEWithLogitsLoss for consistency
criterion = DiceLoss(smooth=1.0)

# Evaluate on validation set
model.eval()
val_loss = 0
val_dice = 0
val_iou_metric = MeanIoU(num_classes=2)
val_iou_metric.to(device)

with torch.no_grad():
    for data, target in tqdm(val_loader, desc="Evaluating"):
        data, target = data.to(device), target.to(device).float()
        output = model(data)
        loss = criterion(output, target)
        val_loss += loss.item()
        val_dice += dice_coeff(output, target).item()
        
        # Calculate IoU
        preds = (torch.sigmoid(output) > 0.5).long()
        targets = target.long()
        val_iou_metric.update(preds, targets)

val_loss /= len(val_loader)
val_dice /= len(val_loader)
val_iou = val_iou_metric.compute().item()

print(f"\nValidation Results (after loading):")
print(f"Loss: {val_loss:.4f}")
print(f"Dice Coefficient: {val_dice:.4f}")
print(f"Mean IoU: {val_iou:.4f}")

# Visualize some predictions
visualize_predictions(model, val_loader, 
                     output_dir="loaded_model_predictions/diceloss/", 
                     device=device,
                     num_samples=5)

/tmp/ipykernel_1129766/3163038322.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Loaded model from models/diceloss/12.pth
Model was trained for 20 epochs
Best validation loss: 0.2535
Best training Dice: 0.8307
Best validation Dice: 0.7465
Best training IoU: 0.7588
Best validation IoU: 0.6413


Evaluating: 100%|██████████| 50/50 [00:03<00:00, 16.36it/s]


Validation Results (after loading):
Loss: 0.2535
Dice Coefficient: 0.7465
Mean IoU: 0.6342


Saved 5 prediction visualizations to loaded_model_predictions/diceloss/
